# Ultra Low Complexity Noise Suppression (ULCNet - Simplified)

This notebook implements an end-to-end simplified version of the ULCNet model
based on the paper "Ultra Low Complexity Deep Learning Based Noise Suppression".

Features:

- STFT + Power Law Compression
- Two-stage Network (CRN + CNN)
- Training + Inference
- Audio Reconstruction


## Install and import libraries


In [1]:

# Install dependencies (run once)
# !pip install torch torchaudio librosa soundfile tqdm


In [1]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from tqdm import tqdm
import os


## Setting up GPU


In [3]:

# GPU Setup and Verification
print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)

# Check and set GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.cuda.empty_cache()
    print("CUDA device initialized and cache cleared")
else:
    print("WARNING: CUDA not available. Using CPU (training will be slow)")

print("=" * 60)


GPU CONFIGURATION
Device: cuda
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA Version: 12.1
GPU Memory: 4.29 GB
CUDA device initialized and cache cleared


## STFT and Power-Law Compression


In [3]:
# ---- Collate function for DataLoader --------------------------------------
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    """
    batch: list of tuples (mix_tensor, clean_tensor) with variable lengths
    Returns:
      mix_padded: [B, Lmax]
      clean_padded: [B, Lmax]
      lengths: tensor([L1, L2, ...])
    """
    mixes = [b[0].float() for b in batch]
    cleans = [b[1].float() for b in batch]
    lengths = torch.tensor([m.shape[0] for m in mixes], dtype=torch.long)

    mix_p = pad_sequence(mixes, batch_first=True)   # [B, Lmax]
    clean_p = pad_sequence(cleans, batch_first=True) # [B, Lmax]

    return mix_p, clean_p, lengths


In [ ]:

def power_compress(x, alpha=0.3):
    """Power-law compression. Operations stay on same device as input."""
    return torch.sign(x) * torch.abs(x) ** alpha

def power_decompress(x, alpha=0.3):
    """Power-law decompression. Operations stay on same device as input."""
    return torch.sign(x) * torch.abs(x) ** (1/alpha)


# ---- Batched STFT / ISTFT helpers ----------------------------------------
# Updated to support batched signals (B, L) or single (L,) - GPU optimized
def compute_stft(wav, n_fft=512, hop=256, win_length=512):
    """
    Compute STFT on GPU.
    wav: [B, L] or [L] tensor on GPU
    Returns: [B, F, T] or [F, T] complex tensor on same device
    """
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    # STFT stays on same device as input (GPU if input is on GPU)
    return torch.stft(wav, n_fft=n_fft, hop_length=hop, win_length=win_length,
                      return_complex=True, center=True, pad_mode='reflect')

def compute_istft(spec, n_fft=512, hop=256, win_length=512, length=None):
    """
    Compute ISTFT on GPU.
    spec: [B, F, T] or [F, T] complex tensor on GPU
    Returns: [B, L] or [L] waveform on same device as input
    """
    # ISTFT stays on same device as input (GPU if input is on GPU)
    return torch.istft(spec, n_fft=n_fft, hop_length=hop, win_length=win_length,
                       length=length, center=True, pad_mode='reflect')


## Stage 1: CRN (Magnitude Mask)


In [5]:

class CRNStage(nn.Module):
    def __init__(self, freq_bins=257):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, (1,3), padding=(0,1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, (1,3), padding=(0,1)),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, (1,3), padding=(0,1)),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.gru = nn.GRU(
            input_size=128*freq_bins,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Sequential(
            nn.Linear(512, freq_bins),
            nn.Sigmoid()
        )


    def forward(self, x):
        # x: [B,1,T,F]
        B,C,T,F = x.shape

        x = self.conv(x)

        x = x.permute(0,2,1,3)
        x = x.reshape(B, T, -1)

        x,_ = self.gru(x)

        mask = self.fc(x)

        return mask


## Stage 2: CNN (Complex Mask)


In [6]:

class CNNStage(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(2, 32, (1,3), padding=(0,1)),
            nn.ReLU(),

            nn.Conv2d(32, 32, (1,3), padding=(0,1)),
            nn.ReLU(),

            nn.Conv2d(32, 2, 1)
        )


    def forward(self, x):
        return self.net(x)


## Full ULCNet Model


In [7]:

class ULCNet(nn.Module):
    def __init__(self, freq_bins=257, alpha=0.3):
        super().__init__()

        self.alpha = alpha

        self.stage1 = CRNStage(freq_bins)
        self.stage2 = CNNStage()


    def forward(self, mag, phase):

        # Stage 1
        mag = mag.unsqueeze(1)
        mask = self.stage1(mag)

        # Intermediate features
        yr = mask * torch.cos(phase)
        yi = mask * torch.sin(phase)

        feat = torch.stack([yr, yi], dim=1)

        # Stage 2
        cmask = self.stage2(feat)

        mr = cmask[:,0]
        mi = cmask[:,1]

        return mr, mi


## Dataset Loader (DNS-Style Simulation)


In [8]:

class SimpleDataset(torch.utils.data.Dataset):

    def __init__(self, clean_files, noise_files, sr=16000):
        self.clean = clean_files
        self.noise = noise_files
        self.sr = sr


    def __len__(self):
        return len(self.clean)


    def __getitem__(self, idx):

        c,_ = librosa.load(self.clean[idx], sr=self.sr)
        n,_ = librosa.load(
            np.random.choice(self.noise),
            sr=self.sr
        )

        L = min(len(c), len(n))
        c = c[:L]
        n = n[:L]

        snr = np.random.uniform(-5,15)
        n = n * 10**(-snr/20)

        mix = c + n

        return torch.tensor(mix), torch.tensor(c)


## Training Loop


In [ ]:
# ---- Updated training loop (masks out padded samples) ---------------------
def train(model, loader, optimizer, device, n_fft=512, hop=256, alpha=0.3):
    model.train()
    total_loss = 0.0
    total_batches = 0

    for mix, clean, lengths in tqdm(loader):
        mix = mix.to(device)       # [B, Lmax] -> GPU
        clean = clean.to(device)   # [B, Lmax] -> GPU
        lengths = lengths.to(device)
        
        # Verify tensors are on GPU
        assert mix.device.type == device.type, f"Mix not on {device}"
        assert clean.device.type == device.type, f"Clean not on {device}"

        # 1) STFT (batched) - GPU operation
        X = compute_stft(mix, n_fft=n_fft, hop=hop)    # [B, F, T] complex on GPU
        
        # Verify STFT output is on GPU
        assert X.device.type == device.type, "STFT output not on GPU"

        # 2) Power-law compress real/imag - GPU operation
        Xr = power_compress(X.real, alpha=alpha)
        Xi = power_compress(X.imag, alpha=alpha)

        # 3) Magnitude and phase - GPU operations
        mag = torch.sqrt(Xr**2 + Xi**2)   # [B, F, T]
        phase = torch.atan2(Xi, Xr)       # [B, F, T]

        mag = mag.permute(0,2,1)   # -> [B, T, F]
        phase = phase.permute(0,2,1) # -> [B, T, F]

        # 4) Forward through model (GPU operation)
        mr, mi = model(mag, phase)  # [B, T, F] each

        # Bring masks back to [B, F, T] to multiply with Xr/Xi
        mr = mr.permute(0,2,1)  # [B, F, T]
        mi = mi.permute(0,2,1)  # [B, F, T]

        # 5) Compose estimated real/imag in compressed domain - GPU operations
        est_r_c = mr * Xr - mi * Xi   # compressed-domain estimated real [B,F,T]
        est_i_c = mr * Xi + mi * Xr

        # 6) Power-decompress to original scale - GPU operations
        est_r = power_decompress(est_r_c, alpha=alpha)
        est_i = power_decompress(est_i_c, alpha=alpha)

        # 7) Reconstruct time-domain signals (GPU operation)
        spec_est = torch.complex(est_r, est_i)  # [B, F, T] on GPU
        Lmax = mix.shape[1]
        out_wav = compute_istft(spec_est, n_fft=n_fft, hop=hop, win_length=n_fft, length=Lmax)  # [B, Lmax] on GPU
        
        # Verify ISTFT output is on GPU
        assert out_wav.device.type == device.type, "ISTFT output not on GPU"

        # 8) Compute MSE loss only over original lengths (GPU operation)
        batch_loss = 0.0
        total_valid = 0
        for i, l in enumerate(lengths):
            li = int(l.item())
            if li == 0:
                continue
            # use reduction='sum' to weight by length - GPU operation
            batch_loss += F.mse_loss(out_wav[i,:li], clean[i,:li], reduction='sum')
            total_valid += li

        if total_valid == 0:
            loss = torch.tensor(0.0, device=device)
        else:
            loss = batch_loss / total_valid

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(1, total_batches)

## Inference / Enhancement


In [ ]:
# ---- Inference: accept (wav, original_length) and crop output (GPU optimized) ----
def enhance(model, wav, device, n_fft=512, hop=256, alpha=0.3):
    """
    Enhance noisy audio on GPU.
    wav: 1D torch tensor or numpy array (variable length)
    device: torch.device for computation
    Returns: enhanced audio clipped to original length
    """
    model.eval()
    if isinstance(wav, np.ndarray):
        wav = torch.tensor(wav, dtype=torch.float32, device=device)  # Create directly on device
    else:
        wav = wav.to(device)  # Move to device if already tensor

    orig_len = wav.shape[0]
    wav_b = wav.unsqueeze(0)  # [1, L] on device
    
    assert wav_b.device.type == device.type, "Input not on GPU"

    with torch.no_grad():
        # All operations now on GPU
        X = compute_stft(wav_b, n_fft=n_fft, hop=hop)  # [1, F, T] complex on GPU

        Xr = power_compress(X.real, alpha=alpha)
        Xi = power_compress(X.imag, alpha=alpha)

        mag = torch.sqrt(Xr**2 + Xi**2).permute(0,2,1)  # [1, T, F] on GPU
        phase = torch.atan2(Xi, Xr).permute(0,2,1)      # [1, T, F] on GPU

        # Forward pass on GPU
        mr, mi = model(mag, phase)  # [1, T, F] each on GPU
        mr = mr.permute(0,2,1)  # [1, F, T]
        mi = mi.permute(0,2,1)

        # GPU operations
        est_r_c = mr * Xr - mi * Xi
        est_i_c = mr * Xi + mi * Xr

        est_r = power_decompress(est_r_c, alpha=alpha)
        est_i = power_decompress(est_i_c, alpha=alpha)

        spec = torch.complex(est_r, est_i)

        # ISTFT on GPU, length ensures proper cropping
        out = compute_istft(spec, n_fft=n_fft, hop=hop, win_length=n_fft, length=orig_len)  # [1, orig_len] on GPU
        out = out.squeeze(0).cpu().numpy()  # Move to CPU only for numpy conversion

    return out  # numpy array (on CPU)

## Example Usage


In [11]:
# training params
clean_dir = "clean_trainset_wav"
noisy_dir = "noisy_trainset_wav"

# testing params
test_file = "app2_data/noisy/p232_003.wav"
output_location = "enhanced_output.wav"

In [ ]:

# Example (requires your own wav files)

# Initialize GPU device (from earlier GPU setup)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create model and move to GPU
model = ULCNet().to(device)
print(f"Model moved to {device}")

# Create optimizer
opt = torch.optim.Adam(model.parameters(), lr=4e-4)

# Prepare dataset
clean_files = []
noise_files = []

# clean files append
for filename in os.listdir(clean_dir):
    if filename.endswith(".wav"):
        clean_files.append(os.path.join(clean_dir, filename))
        
# noise files append
for filename in os.listdir(noisy_dir):
    if filename.endswith(".wav"):
        noise_files.append(os.path.join(noisy_dir, filename))
  
# Uncomment for quick testing
# clean_files = clean_files[:5]
# noise_files = noise_files[:5]

print(f"Found {len(clean_files)} clean files and {len(noise_files)} noise files")
    
# ---- DataLoader usage: pass collate_fn -----------------------------------
dataset = SimpleDataset(clean_files, noise_files, sr=16000)
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0   # IMPORTANT: Keep at 0 for CUDA on Windows
)

print("DataLoader created successfully")
print(f"Total batches per epoch: {len(loader)}")


In [ ]:
# Train with GPU monitoring
import time

print("="*60)
print("TRAINING ON GPU")
print("="*60)

for epoch in range(10):
    start_time = time.time()
    loss = train(model, loader, opt, device)
    epoch_time = time.time() - start_time
    
    print(f"Epoch {epoch:2d} | Loss: {loss:.6f} | Time: {epoch_time:.2f}s")
    
    # Save model
    torch.save(model.state_dict(), f"ulcnet_model_{epoch}.pth")
    
    # Monitor GPU
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"         GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")
        
print("="*60)
print("Training Complete!")
print("="*60)


  0%|          | 0/2893 [00:00<?, ?it/s]c:\Users\Aryan Gupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\functional.py:709: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window can are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\SpectralOps.cpp:842.)
  return _VF.stft(  # type: ignore[attr-defined]
C:\Users\Aryan Gupta\AppData\Local\Temp\ipykernel_29260\3557808025.py:24: UserWarning: A window was not provided. A rectangular window will be applied.Please provide the same window used by stft to make the inversion lossless.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Trig

Epoch 0 0.0016322955575021468


100%|██████████| 2893/2893 [1:13:17<00:00,  1.52s/it]


Epoch 1 0.0016487794210063116


100%|██████████| 2893/2893 [1:15:40<00:00,  1.57s/it]


...

Epoch 4 0.0016541382084157019


100%|██████████| 2893/2893 [1:16:48<00:00,  1.59s/it]


Epoch 5 0.0016262919066674722


 45%|████▌     | 1310/2893 [34:32<41:44,  1.58s/it] 


KeyboardInterrupt: 

## Save Model


In [13]:
# add a way to save and load the model
# torch.save(model.state_dict(), "ulcnet_model.pth")
# Load the model (example)
model.load_state_dict(torch.load("ulcnet_model_1.pth"))

<All keys matched successfully>

In [ ]:
# Enhance with GPU
print("Enhancing audio on GPU...")
wav, _ = librosa.load(test_file, sr=16000)
start_time = time.time()
out = enhance(model, torch.tensor(wav), device)  # Use GPU for enhancement
inference_time = time.time() - start_time

sf.write(output_location, out, 16000)
print(f"Enhanced audio saved to {output_location}")
print(f"Inference time: {inference_time:.3f}s")


c:\Users\Aryan Gupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\functional.py:709: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window can are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\SpectralOps.cpp:842.)
  return _VF.stft(  # type: ignore[attr-defined]
C:\Users\Aryan Gupta\AppData\Local\Temp\ipykernel_14464\3557808025.py:24: UserWarning: A window was not provided. A rectangular window will be applied.Please provide the same window used by stft to make the inversion lossless.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at C:\actions-runner\_w

## Test set


In [13]:
# test set directory location
test_noise_dir = "noisy_testset_wav"
test_clean_dir = "clean_testset_wav"

In [14]:
# load filenames
test_noisy_files = []
test_clean_files = []

for filename in os.listdir(test_noise_dir):
    if filename.endswith(".wav"):
        test_noisy_files.append(os.path.join(test_noise_dir, filename))
for filename in os.listdir(test_clean_dir):
    if filename.endswith(".wav"):
        test_clean_files.append(os.path.join(test_clean_dir, filename))

In [ ]:
# define error function (GPU optimized)
def compute_mse_on_test_set(model, noisy_files, clean_files, device):
    """Compute MSE on test set using GPU for all computations."""
    model.eval()
    total_mse = 0.0
    total_samples = 0

    with torch.no_grad():
        for nfile, cfile in zip(noisy_files, clean_files):
            noisy_wav, _ = librosa.load(nfile, sr=16000)
            clean_wav, _ = librosa.load(cfile, sr=16000)

            # Enhance on GPU
            enhanced_wav = enhance(model, torch.tensor(noisy_wav), device)

            L = min(len(enhanced_wav), len(clean_wav))
            # Compute MSE
            mse = np.mean((enhanced_wav[:L] - clean_wav[:L])**2)

            total_mse += mse * L
            total_samples += L

    overall_mse = total_mse / total_samples if total_samples > 0 else 0.0
    return overall_mse


In [16]:
# ouput error on test set
mse_error = compute_mse_on_test_set(model, test_noisy_files, test_clean_files, device)
print("MSE on test set:", mse_error)

c:\Users\Aryan Gupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\functional.py:709: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window can are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\SpectralOps.cpp:842.)
  return _VF.stft(  # type: ignore[attr-defined]
C:\Users\Aryan Gupta\AppData\Local\Temp\ipykernel_29252\3557808025.py:24: UserWarning: A window was not provided. A rectangular window will be applied.Please provide the same window used by stft to make the inversion lossless.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at C:\actions-runner\_w

MSE on test set: 0.0007196158471051352


In [ ]:
# Comprehensive GPU performance check
print("\n" + "="*60)
print("GPU PERFORMANCE SUMMARY")
print("="*60)
print(f"Device: {device}")
print(f"Model location: {next(model.parameters()).device}")
print(f"Total model parameters: {sum(p.numel() for p in model.parameters()):,}")

if torch.cuda.is_available():
    print(f"\nGPU Memory:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"  Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"  Cached:    {torch.cuda.memory_cached() / 1e9:.2f} GB")
    
    # Memory breakdown
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1e9:.2f} GB")
        print(f"  Used: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
        
print("\n✓ All computations are running on GPU")
print("="*60)


## Notes

This is a faithful but simplified implementation for learning and experimentation.

For production, full subband GRUs and channelwise reorientation should be added.
